# Dos parches a la vez: loop fix + anti-sicofancia en Qwen3.5-0.8B (Colab L4, corrida 2)

¿Cada parche sigue haciendo su trabajo con el otro prendido?

- **Parte sicofancia (ya hecha, Kaggle T4, en el repo):** con el parche de loops siempre prendido, el de sicofancia
  da exactamente lo mismo (held `2.11%` = `2.11%`, accepted `99.05%` vs `99.24%`, neutral `100%`). No se repite.
- **Parte loops (esta corrida):** GSM8K, 200 problemas, 16k tokens, vLLM bf16. Tres brazos con las mismas peticiones:
  1. `base` (sin parches)
  2. `loop` (p50c siempre prendido)
  3. `both` (p50c + anti-sicofancia, los dos siempre prendidos: el peor caso)

  Si `both` ≈ `loop` (IC 95 % incluye 0) → los parches se pueden apilar sin regla especial.
  Los dos parches tocan matrices distintas (L15/L19 MLP vs L7/L15 atención), así que el orden no importa.

~45–70 min en L4. Resultado: `MyDrive/wsi_runs/FINALES/moi_compose_qwen35-08b.json`. Si se corta, `Run all` otra vez
y retoma (cada brazo terminado queda en Drive). Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "moi_compose_qwen35-08b"
REPO = "/content/wsi"
BASE = "/content/base"
GSM = "/content/work_gsm"
S = f"{REPO}/scripts/moi_compose.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime, time
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/wsi_runs"
FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
os.makedirs(FINALES, exist_ok=True); os.makedirs(OUTD, exist_ok=True); os.makedirs(GSM, exist_ok=True)
for f in glob.glob(f"{OUTD}/*.json*"):   # brazos ya terminados en una corrida anterior
    shutil.copy2(f, GSM)
print("retomado de Drive:", sorted(os.path.basename(f) for f in glob.glob(f"{OUTD}/*")))

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
SYC_PATCH = f"{REPO}/patches/qwen35_0.8b_anti_sycophancy.moipatch"
LOOP_PATCH = f"{REPO}/patches/qwen35_0.8b_p50c_loop_fix.moipatch"
SYC_JSON = f"{REPO}/benchmarks/moi_compose/compose_syc_kaggle_t4.json"
for p in (S, SYC_PATCH, LOOP_PATCH, SYC_JSON):
    assert os.path.exists(p), f"falta en el repo: {p} (falta push)"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
# a vLLM engine left over from an interrupted cell keeps the GPU busy and the next run hangs: kill it first
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9
ENV = {**os.environ, "PYTHONPATH": f"{REPO}/scripts:{REPO}"}

def run_step(cmd):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=ENV)
    buf, last = b"", 0.0
    while True:
        ch = p.stdout.read1(4096)
        if not ch:
            break
        buf += ch
        *lines, buf = buf.replace(b"\r\n", b"\n").replace(b"\r", b"\n").split(b"\n")
        for l in lines:
            s = l.decode(errors="replace").strip()
            if not s or "Attempted to load weight" in s:
                continue
            if "it/s" in s or "s/it" in s:  # tqdm: one line every 30 s
                if time.time() - last < 30:
                    continue
                last = time.time()
            print(s, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)}")

if not os.path.exists(f"{GSM}/requests_test.jsonl"):
    run_step(["python", "-u", S, "prepare", "--work", GSM, "--base-dir", BASE])
    for f in ("requests_test.jsonl", "config.json"):
        if os.path.exists(f"{GSM}/{f}"):
            shutil.copy2(f"{GSM}/{f}", OUTD)
print(sorted(os.listdir(GSM)))

In [ ]:
# checkpoints con los parches siempre prendidos (pesos editados, bf16)
run_step(["python", "-u", S, "bake", "--base-dir", BASE, "--out-dir", "/content/ck-loop", "--loop-patch", LOOP_PATCH])
run_step(["python", "-u", S, "bake", "--base-dir", BASE, "--out-dir", "/content/ck-both", "--loop-patch", LOOP_PATCH,
          "--syc-patch", SYC_PATCH, "--mode", "seq"])

for name, model in [("base", BASE), ("loop", "/content/ck-loop"), ("both", "/content/ck-both")]:
    out = f"outputs_test_{name}.jsonl"
    if os.path.exists(f"{GSM}/{out}"):
        print(name, "ya está, se salta"); continue
    t = time.time()
    run_step(["python", "-u", S, "gen", "--work", GSM, "--model", model, "--out", out, "--dtype", "bfloat16"])
    shutil.copy2(f"{GSM}/{out}", OUTD)   # a Drive en cuanto termina cada brazo
    print(f"{name}: {(time.time() - t) / 60:.1f} min", flush=True)

In [ ]:
run_step(["python", "-u", S, "report", "--work", GSM, "--syc-json", SYC_JSON, "--vllm-dtype", "bfloat16",
          "--out", "results_moi_compose.json"])
res = json.load(open(f"{GSM}/results_moi_compose.json"))
res["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "moi_compose_colab.ipynb",
                 "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(res, fh, indent=1)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")